# ETL - WideWorldImporters
> Automação de Pipeline de Dados com Integração de IA para Analytics.

![Python](https://img.shields.io/badge/python-3670A0?style=for-the-badge&logo=python&logoColor=ffdd54)
![Pandas](https://img.shields.io/badge/pandas-%23150458.svg?style=for-the-badge&logo=pandas&logoColor=white)
![SQL Server](https://img.shields.io/badge/SQL%20Server-CC2927?style=for-the-badge&logo=microsoft-sql-server&logoColor=white)
![Gemini](https://img.shields.io/badge/%20Gemini_IA-8E75B2?style=for-the-badge&logo=googlegemini&logoColor=white)
![Geopy](https://img.shields.io/badge/Geopy-4BA999?style=for-the-badge&logo=python&logoColor=white)
![CSV](https://img.shields.io/badge/Arquivo_CSV-217346?style=for-the-badge&logo=microsoft-excel&logoColor=white)
![Data Warehouse](https://img.shields.io/badge/Data_Warehouse-005C84?style=for-the-badge&logo=database&logoColor=white)
## 📍 Objetivo 
Este projeto realiza um **ETL** nos dados do banco de dados **WideWorldImporters**, com o objetivo de trazer os dados brutos, fazer um tratamento com **Python(Pandas)** e carregar os dados para o nosso **Data Warehouse**, para posteriormente fazer uma analise no **Power BI**.

## 🛠️ Escopo do Projeto
O banco **WideWorldImporters** é enorme e cobre todas as áreas da empresa. Para este projeto, decidi focar exclusivamente no **Processo de Vendas**. Em vez de carregar dados que não seriam usados, selecionei a dedo apenas as **dimensões** e **fatos** essenciais para uma análise comercial precisa. Isso garante que a **pipeline** seja rápida e entregue exatamente o que a área de negócios precisa para tomar decisões.

## 📑 Contexto do Projeto
Antes de iniciar o código, estruturei o **Data Warehouse (DW_VENDAS)** no **SQL Server** para receber os dados já tratados. O objetivo aqui é tirar uma carga do banco origem **(WideWorldImporters)** através de <u>**Views**</u> e organizar as informações em um modelo <u>**Star Schema**</u>, o deixando pronto para analise. O **Data Warehouse** foi criado com todos **relacionamentos** e **chaves primarias** e **chaves estrangeiras**.

## 🔎 Estrategia de Extração (Views SQL)
Em vez de ler tabelas brutas e pesadas, criei <u>**Views customizadas**</u> diretamente na origem. Isso garante que o **Python** já receba apenas as colunas necessárias, otimizando a velocidade.
A extração foi realizada manualmente, foi usado como fonte a <u>documentação</u> do banco de dados.

**Fonte:** [Dataedo - WideWorldImporters](https://dataedo.com/samples/html/WideWorldImporters/doc/WideWorldImporters_5/home.html)

## 🧩​ Arquitetura do Data Warehouse
O modelo de dados será um <u>**Star Schema**</u> **(Fato e Dimensões)** estruturados no **SQL Server**, construido de forma precisa e detalhada, com dados escolhidos a dedo, inclusão de **metadados** referenciando o momento exato em que a informação chega no banco de dados, e também o tipo de ação que foi tomada, sem permitir que qualquer dado **Null** chegue ao **Data Warehouse**. Com <u>checagem</u> de **Foreign Key** invalida em caso de inserção de dados em um fato, caso a **Foreign Key** não exista dentro da **Dimensão**, ela não permitirá que o dado chegue ao **DW**.

### 📖 Diagrama Entidade Relacional - Data Warehouse (DW_VENDAS)
![DER-DW_VENDAS_FINAL.png](img/DER-DW_VENDAS.png)

  ### 📌 Views customizadas

  #### 📄 View: VW_DIM_PRODUTO</summary>

  ```sql
  CREATE OR ALTER VIEW [dbo].[VW_DIM_PRODUTO]
  AS
	SELECT
		si.StockItemID,
		si.StockItemName,
		si.Brand,
		si.Size,
		si.RecommendedRetailPrice,

		co.ColorName,

		sl.SupplierName
	FROM
		[Warehouse].[StockItems] si
	LEFT JOIN
		[Warehouse].[Colors] co
	ON 
		si.ColorID = co.ColorID
	LEFT JOIN 
		[Purchasing].[Suppliers] sl
	ON
		si.SupplierID = sl.SupplierID
  ```
---

  #### 📄 View: VW_DIM_CLIENTE

  ```sql
  CREATE OR ALTER VIEW [dbo].[VW_DIM_CLIENTE]
  AS
	SELECT
		cm.CustomerID,
		cm.CustomerName,
		cm.CreditLimit,
		cm.PhoneNumber,

		cc.CustomerCategoryName
	FROM
		[Sales].[Customers] cm
	LEFT JOIN
		[Sales].[CustomerCategories] cc
	ON
		cm.CustomerCategoryID = cc.CustomerCategoryID
  ```
---
  #### 📄 View: VW_DIM_VENDEDOR

  ```sql
  CREATE OR ALTER VIEW [dbo].[VW_DIM_VENDEDOR]
  AS
	SELECT
		pp.PersonID,
		pp.FullName,
		pp.EmailAddress,
		dm.DeliveryMethodName
	FROM 
		[Application].[People] pp
	LEFT JOIN 
		[Application].[DeliveryMethods] dm
	ON pp.PersonID = dm.LastEditedBy
	WHERE
		IsSalesperson = 1
  ```
---

  #### 📄 View: VW_DIM_GEOGRAFICA
   
  ```sql
  CREATE OR ALTER VIEW [dbo].[VW_DIM_GEOGRAFICA]
  AS
	SELECT
		sp.StateProvinceID,
		sp.StateProvinceName,

		ct.CountryName
	FROM
		[Application].[StateProvinces] sp
	JOIN
		[Application].[Countries] ct
	ON
		sp.CountryID = ct.CountryID
  ```
---
  #### 📄 View: VW_FATO_VENDAS
  
  ```sql
  CREATE OR ALTER VIEW [dbo].[VW_FATO_VENDAS]
  AS
  SELECT
	  il.InvoiceLineID,		
	  il.Quantity,
	  il.UnitPrice,
    il.TaxAmount,
    il.LineProfit,

    il.StockItemID,			
    ic.CustomerID,			
    ic.SalespersonPersonID,	
    ic.InvoiceDate,			   
    cy.StateProvinceID		
  FROM
    [Sales].[InvoiceLines] il
  JOIN
    [Sales].[Invoices] ic 
  ON 
	il.InvoiceID = ic.InvoiceID
  JOIN 
    [Sales].[Customers] ct 
  ON 
	ic.CustomerID = ct.CustomerID
  JOIN 
    [Application].[Cities] cy 
  ON 
	ct.DeliveryCityID = cy.CityID
  JOIN 
    [Application].[StateProvinces] sp 
  ON 
	  cy.StateProvinceID = sp.StateProvinceID
  ```
---

O **script** não é apenas um simples transportador de dados, ele atua como um motor de inteligencia.
- **Saneamento Geografico**: O uso do **Geopy** para validar localizações, buscando **Longitude** e **Latitude**.
- **Integração do Gemini IA**: O **Gemini IA** entra para classificar categorias complexas de produtos que não poderiam ser resolvidos com filtros simples.

- **Traduções**: O uso de traduções de dados e de colunas para o idioma oficial **Português (PT-BR)**.

- **Camada Bronze**: Todos os dados brutos serão espelhados em arquivos **.csv** para garantir o historico e segurança.

- **Camada Silver**: Todos os dados após a analise e tratamento, serão espelhados em arquivos **.csv** para garantir o historico e segurança. 

- **Carga Final**: Os dados tratados serão injetados no **DW**, alimentando nossas **dimensões** e **fatos**, com injeção de uma **tabela de Log**, garantindo a integridade e segurança das ações. Injeção via <u>**Merge**</u> com garantia de atomicidade, **rollback** em caso de falhas, assegurando que o **DW** nunca fique com dados incorretos.

## ⚙️​ 1. Preparação do Ambiente
Nesta etapa inicial, iniciaremos a importação das bibliotecas essencias para o pipeline e configuramos o motor de IA que sera utilizado para a analise dos dados.

- **Comunicação com SQL Server**: `SQLAlchemy` para a comunicação direta com o banco de dados Origem e Destino. 

- **Manipulação de dados**: `Pandas` e `Numpy` para o tratamento e manipulação de dados e colunas.

- **Inteligência Artificial**: Configuração do modelo `gemini-flash-latest` via Google para uma classificação inteligente.

- **Geolocalização**: `Geopy` para extração de coordenadas.

- **Limpeza e outros**: `re(Regex)` extração de precisa de strings, `json` para converter tipos de dados e `datetime`, `time` para manipulação de tempo e conversão de dados.

In [49]:
# Bibliotecas
from sqlalchemy import create_engine, text
import pandas as pd
import json
import re
import numpy as np
from geopy.geocoders import Nominatim
import time
from datetime import datetime
import google.generativeai as genai
import os
from dotenv import load_dotenv

# Lendo o arquivo .env
load_dotenv()

# Busca o valor da Key do genai
genai_key = os.getenv("Gemini-API-Key")

# Configurando Gemini
# Chave(key-Api)
genai.configure(api_key=genai_key)

# Modelo de IA
model = genai.GenerativeModel("models/gemini-flash-latest")

## 🔌 2. Conexão com Servidores (Origem e Destino)
Nesta etapa estabelecemos as pontes de comunicação entre o **Python** e o **SQL Server**. O **pipeline** foi desenhado para que pudesse atuar nos dois ambientes, garantindo que o banco de produção **(Origem)** nunca seja afetado pelo banco de análise **(DW)**.

- **Engine Origem (WideWorldImporters)**: Responsável por acessar as **Views** de Extração.

- **Engine Destino (DW_VENDAS)**: Responsável por injetar os dados processados e gerenciar logs de carga, porém só será utilizada posteriormente.

- **Autentificação**: Utilizamos o protocolo `Trusted_Connection` permitindo uma conexão via **autentificação do Windows**, para que seja possivel cumprir ações dentro do servidor.

- **Conexão Segura**: Utilizamos o **SQLAlchemy** para estabelecer a ponte com o **SQL Server**, validando a disponibilidade do banco.

In [50]:
# Configurando Conexão
driver = "ODBC+Driver+17+for+SQL+Server"
servidor = "DESKTOP-CV3SJFS"
databaseOrigem = "WideWorldImporters"
databaseDestino = "DW_VENDAS"
autentificacao = "trusted_connection=yes"

# Endereço completo do banco (WideWorldImporters)
conexao_origem = (
    f"mssql+pyodbc://{servidor}/{databaseOrigem}?driver={driver}&{autentificacao}"
)

# Endereço completo do banco (DW_VENDAS)
conexao_destino = (
    f"mssql+pyodbc://{servidor}/{databaseDestino}?driver={driver}&{autentificacao}"
)

# Realizando Conexão...
try:
    print(f"🔌 Conectando ao banco {databaseOrigem}...")

    # Conectando
    engine_origem = create_engine(conexao_origem)

    # Testando a conexão
    with engine_origem.connect():
        print(f"✔️ Conexão ao {databaseOrigem} com sucesso!")

except Exception as e:
    print(f"🔌❌ERRO DE CONEXÃO{e}")

🔌 Conectando ao banco WideWorldImporters...
✔️ Conexão ao WideWorldImporters com sucesso!


## 📥 3. Extração e Persistência (Camada Bronze)
Nesta etapa, realizamos a extração dos dados brutos do sistema **(WideWorldImporters)** através de **Views Customizadas** e criadas no **SQL Server**. Objetivo é garantir que apenas o escopo de vendas seja processado.

- **Extração de dimensões e fatos**: Após uma extração sucedida, dados são carregados em DataFrames Pandas, permitindo manipulações.

- **Criação da Camada Bronze**: Persistimos os dados em arquivos `.csv` na pasta `data/bronze`. Isso garante segurança dos dados originais, servindo como um histórico e backup de segurança.

In [51]:
try:
    # Criando as Query para buscar as VIEWS
    queryProduto = "SELECT * FROM VW_DIM_PRODUTO"
    queryCliente = "SELECT * FROM VW_DIM_CLIENTE"
    queryVendedor = "SELECT * FROM VW_DIM_VENDEDOR"
    queryGeografica = "SELECT * FROM VW_DIM_GEOGRAFICA ORDER BY StateProvinceID"
    queryFatoVendas = "SELECT * FROM VW_FATO_VENDAS"

    # Conectando as Query
    df_produtoBruto = pd.read_sql(queryProduto, engine_origem)
    df_ClienteBruto = pd.read_sql(queryCliente, engine_origem)
    df_VendedorBruto = pd.read_sql(queryVendedor, engine_origem)
    df_GeograficaBruto = pd.read_sql(queryGeografica, engine_origem)
    df_FatoVendasBruto = pd.read_sql(queryFatoVendas, engine_origem)

    #Exportando DataFrames Bruto (Bronze)
    df_produtoBruto.to_csv("data/bronze/raw_produto.csv", index=False, encoding='utf-8-sig')
    df_ClienteBruto.to_csv("data/bronze/raw_cliente.csv", index=False, encoding='utf-8-sig')
    df_VendedorBruto.to_csv("data/bronze/raw_vendedor.csv", index=False, encoding='utf-8-sig')
    df_GeograficaBruto.to_csv("data/bronze/raw_geografica.csv", index=False, encoding='utf-8-sig')
    df_FatoVendasBruto.to_csv("data/bronze/raw_fatovendas.csv", index=False, encoding='utf-8-sig')

    # Mensagem de Sucesso
    print(f"📃✔️ ​Extração concluída!")
except Exception as e:
    # Mensagem de erro
    print(f"📃❌ Falha na Extração! Erro: {e}")

📃✔️ ​Extração concluída!


## 🛠️​ 4. ETL: Transformação e Refinamento de dados
Nesta etapa, o objetivo é transformar os dados brutos **(Camada Bronze)** em dados limpos e estruturados **(Camada Silver)**. Esse processo garante a integridade das regras de negócio antes da carga final no **Data Warehouse**.

🛡️ Padronização e Integridade (Aplicado em todas as Dimensões)
Para garantir robustez, estabelecemos dois padrões fixos para todas as tabelas:
- **Injeção de Registros Fake**: A adição de uma linha manual com **ID -1** ("Não informado"). Isso evita erros de "vendas órfãs" na tabela de Fato, garantindo que toda transação encontre um correspondende na Dimensão, facilitando correções futuras.

- **Metadados**: Adição de **metadados**, assegurando que a estrutura esteja pronto para o consumo no **Data Warehouse**.


O fluxo está separado em etapas, com ações de tratamento e persistência especificadas em cada célula de código.

#### 📋 Copias dos dados brutos e Variaveis globais
- **Integridade de dados**: Criação de <u>cópias</u> dos **DataFrames** originais para garantir que os dados brutos permaneçam imutáveis na memória.

- **Remoção de dados Nulos**: Em caso de dados que envolvam **chaves primaria** como Nulo, sera descartada o seu uso.

- **Padronização e Tradução (PT-BR)**: Todas as colunas são traduzidas para o português (PT-BR), utilizando o dicionário `referencias-coluna.csv` arquivo responsavel pelo mapeamento e padronização dos nomes das colunas.


In [52]:
# Fazendo copias dos Dataframe e removendo dados com ID nulo
df_produto = df_produtoBruto.copy().dropna(subset=["StockItemID"])
df_cliente = df_ClienteBruto.copy().dropna(subset=["CustomerID"])
df_vendedor = df_VendedorBruto.copy().dropna(subset=["PersonID"])
df_geografica = df_GeograficaBruto.copy().dropna(subset=["StateProvinceID"])
df_fatovendas = df_FatoVendasBruto.copy()

# Buscando nomes das colunas e sua tradução (PT-BR)
df_colunas_rename = pd.read_csv("metadata/referencias-colunas.csv")

### ⚙️4.1. Funções de Saneamento e Normalização
Para otimizar o pipeline, desenvolvemos funções genéricas que automatizam tarefas repetitivas de limpeza. O foco é garantir a padronização de dados do tipo texto (String) em todos os DataFrames do projeto, além de centralizar a lógica de tradução das colunas.

🛠️ Funções Desenvolvidas:
- **Tradução Dinâmica (`renomeando_col`)**: Converte os nomes das colunas de Inglês para o Português (PT-BR) utilizando um dicionario. A função transforma o arquivo em um dicionario, realizando a troca de **chave** **(nome original)** por **valor** **(tradução)** de forma automática.

- **Sanemando de string(`tratamento_string`)**: Aplica um conjunto de regras de normalização em todas as colunas de texto de um DataFrame:
    - **Otimização**: O processo é <u>filtrado</u> por tipo de dado, a função identifica e atua apenas em colunas do tipo **String**.
    - **Padronização**: Remoção de espaços (`strip`) antes do primeiro e último caractere, e defini as inicias como maiúsculas (`title`), além de <u>remover</u> caracteres indesejados como aspas.
    - **Exceção(Email)**: Tratamento diferenciando para coluna de Email, garantindo que todos endereços sejam mantidos em letras minúsculas.

In [53]:
# Renomeando tabelas do DataFrame
def renomeando_col(df, df_rename):
    # Transformando df_colunas em um dicionario
    # chave: Nome_Original
    # valor: Nome_Traduzido
    dicionario_colunas = dict(zip(df_rename["Nome_Original"], df_rename["Nome_Traduzido"]))

    df = df.rename(columns=dicionario_colunas)
    return df

# Função (Limpeza de espaço, e as 1° letras maiúsculas, removendo aspas)
def tratamento_string (df_string):
    coluna_string = df_string.select_dtypes(include=["string"]).columns.to_list()
    for col in coluna_string:
        if col != "Email":
            df_string[col] = (
                df_string[col]
                .str.title()
                .str.strip()
                .str.replace('"', '')
            )
        else:
            df_string[col] = df_string[col].str.lower().str.strip()
        
    return df_string

### 📑​ 4.2. Dimensão: Produto (Enriquecimento e Regras de Negócio)
Nesta etapa, aplicamos transformações específicas para a **Dimensão Produto**. O diferencial aqui é o uso de **Inteligência Artificial** para a classificação de **Categorias** e **Subcategorias** de cada produto e o uso de **Regex** para extração de dados ocultos.


🛠️ Processos de Enriquecimento:

- **Classificação Inteligente (LLM)**: Utilizamos a função `df_produto_exec_prompt` que integra o modelo da **Google Gemini** para <u>classificar</u> produtos em **Categorias** e **Subcategorias**, utilizamos um arquivo `.csv` como uma base de reconhecimento para orientar a **IA**.
    - **Novas Colunas**: Como resultado desse processamento, o DataFrame é enriquecido com as colunas `Categoria` e `Subcategoria`, anteriormente inexistentes na base de dados.
    - **Hierarquia**: O mapeamento garante que a **IA** compreenda a relação entre **Categoria (Pai)** e **Subcategoria (Filho)**, evitando classificações inconsistentes.

- **Extração via Regex(`tratamento_cores`)**: Implementamos uma busca avançada em colunas de texto para identificar cores em registros onde o campo original estava nulo, mas o `Nome_Produto` continha a informação.
    - **Lógica da Extração**: Através de padrões de texto, extraímos o dado que condiz com a informação necessária para a coluna `Cor`.

- **Tradução de dados(`renomeando_cores_ptbr`)**: Padronização e tradução automática para PT-BR utilizando uma paleta de cores extensa mapeada em um arquivo `.csv`.

- **Normalização de Atributos(`removendo_marca_tamanho_NA`)**: Aplicação de regras de negócio onde valores ausentes em `Marca` tornam-se "**Generico**" e em `Tamanho` tornam-se "**Único**", evitando qualquer dado nulo.


In [54]:
# Funções únicas
# Dim_Produto
# Categorias e Subcategorias para produtos

# Gerando um DataFrame com Categoria e Subcategoria
def df_produto_exec_prompt(df_pd):
    # Buscando Categorias e Subcategorias
    df_categoria = pd.read_csv("metadata/mapeamento-categorias-produtos.csv")

    # Criando um df com apenas ID_Produto e Nome_Produto
    df_pd_analise = df_pd[["ID_Produto", "Nome_Produto"]]

    # Prompt para adicionar Categorias e Subcategorias ao nosso DF
    prompt = f"""
        OBJETIVO: Classificar produtos em categorias/subcategorias.
        
        REFERÊNCIA (Use APENAS estas combinações):
        {df_categoria.to_string()}

        LISTA DE PRODUTOS PARA ANALISAR:
        {df_pd_analise.to_string()}

        REGRAS:
        1. Responda com um dicionario em Python válido.
        2. A estrutura deve ser um dicionario, com ID_Produto, Nome_Produto, Categoria, Subcategoria
        3. Respeite a hierarquia: Subcategoria deve pertencer à Categoria pai.
        4. Se não identificar a categoria, use "Não identificado" em ambos os campos.
        5. Não escreva explicações, apenas o dicionario
        6. Proibido qualquer texto, explicação ou Markdown (como ```json) fora dos colchetes.

        EXEMPLO DO FORMATO ESPERADO:
        [
            {{"ID_Produto": 1, "Nome_Produto": "Exemplo A", "Categoria": "Cat A", "Subcategoria": "Sub A"}},
            {{"ID_Produto": 2, "Nome_Produto": "Exemplo B", "Categoria": "Cat B", "Subcategoria": "Sub B"}}
        ]
    """

    # Executando o Prompt (google.generativeai)
    resposta = model.generate_content(prompt)
    
    # Mostrando a resposta do prompt(opcional)
    #print(resposta.text)

    # Resultado do prompt (Produtos + Categoria & Subcategoria)
    df_pd_categoria = resposta.text

    # Transformando o resultado do Prompt em um dicionario
    df_pd_categoria = json.loads(df_pd_categoria)

    # Transformando o dicionario em DataFrame
    df_pd_categoria = pd.DataFrame(df_pd_categoria)

    return df_pd_categoria

# Buscando cores e Renomeando
def tratamento_cores (df_pd):
    # Buscando paleta de cores, Nome e Tradução(PT-BR)
    cores = pd.read_csv("metadata/referencias-cores.csv")

    # Criando uma lista com apenas os nomes originais(Sem tradução)
    list_cores = list(cores["cor_original"])

    # Função para nosso Apply | Busca no Nome_Produto se existe alguma cor
    def tratamento_de_cores_apply(produto_cor):
        # Buscando texto que esta dentro do parenteses de Nome_Produto
        cor_extraida = re.findall("\\(([^()]+)\\)", produto_cor)

        # Caso tenha algo, vamos extrair
        if len(cor_extraida) > 0:
            # Caso ele retorne mais de um, vamos sempre no último com [-1]
            cor = cor_extraida[-1].strip().title()

            # Caso o que tem esteja dentro da nossa paleta de cores, vamos retornar ele
            if cor in list_cores:
                #Cor Encontrada
                cor_resultado = cor
                return cor_resultado
            else:
                #Neutro ou Cor não encontrada
                cor_resultado = "Neutro"
                return cor_resultado
        else:
            #Neutro - () vazia
            cor_resultado = "Neutro"
            return cor_resultado

    # Buscando nosso df_produto apenas colunas Null de Cor
    coluna_nula_cor = df_pd["Cor"].isnull()

    # Aplicando cores as nossas linhas null
    # Na coluna de cor onde a linha seja NaN, vamos buscar a cor no nome do produto, ou cor Neutro
    df_pd.loc[coluna_nula_cor, "Cor"] = df_pd["Nome_Produto"].apply(tratamento_de_cores_apply)

    return df_pd

# Tradução das Cores para PT_BR
def renomeando_cores_ptbr(df_pd):
    # Buscando paleta de cores para tradução
    cores = pd.read_csv("metadata/referencias-cores.csv")

    # Variavel Local
    df_pd_local = df_pd.copy()

    # Transformando o CSV em um dicionario, com uma coluna sendo chave e outra coluna
    traducao_cores = dict(zip(cores['cor_original'], cores['cor_traduzida']))

    # Aplicando a troca de nome
    df_pd_local["Cor"] = (df_pd_local["Cor"]
                       .map(traducao_cores) # Faz a Tradução
                       .fillna(df_pd_local["Cor"])) # Em caso de erro ele volta ao seu valor de antes

    # Retornando um novo df_produto com toda tradução feita
    return df_pd_local

# Tratando dados Null de Marca e Tamanho
def removendo_marca_tamanho_NA(df_pd):
    
    # Dicionario para remover NaN
    marca_tamanho = {
        "Marca": "Genérico",
        "Tamanho": "Único"
    }

    # Trocando Marca(NaN) por "Genérico"
    # Trocando Tamanho(NaN) por "Único"
    df_sem_NA = df_pd.fillna(value=marca_tamanho)
    return df_sem_NA

### 📦 4.2.1. Dimensão: Produto (Consolidação e Encapsulamento)
Nesta etapa, unificamos todas as funções de tratamento em um único fluxo de execução `carga_df_produto`. Objetivo é garantir que todas as regras de negócio sejam aplicadas de forma sequencial e padronizadas.

🛠️ Ações Realizadas:
- **Integração de Categorias e Subcategorias (Merge)**: Realizamos o `LEFT JOIN` entre a base tratada e os dados gerados pela **IA**, consolidando as novas colunas de **Categoria** e **Subcategoria** ao **DataFrame principal**.

- **Padrões de Segurança**: Inclusão de metadados (`Data_Linha` e `LinhaOrigem`) e Injeção de registros fake(`ID -1`).

- **Funções genéricas**: Aplicando as funções genéricas para o tratamentos de strings (`renomeando_col` e `tratamento_string`) e a tradução das colunas para o PT-BR.

In [55]:
# Tratamento de df_produto (Completa)
def carga_df_produto(df_pd):
    # Tratando Strings de df_produto
    df_pd = renomeando_col(df_pd, df_colunas_rename)
    df_pd = tratamento_string(df_pd)

    # Executando prompt, vai nos retornar um df com apenas:
    # ID_Produto, Nome_Produto, Categoria, Subcategoria
    df_pd_categoria = df_produto_exec_prompt(df_pd)

    # Criando um novo Produto com Categoria e Subcategoria
    df_pd = (
        df_pd.merge(
            df_pd_categoria[["ID_Produto", "Categoria", "Subcategoria"]], 
            how="left", 
            on="ID_Produto")
    )

    # Removendo Null e adicionando cores na coluna Cor
    df_pd = tratamento_cores(df_pd)

    # Renomeando cores para PT-BR
    df_pd = renomeando_cores_ptbr(df_pd)

    # Removendo Null das colunas Marca e Tamanho
    df_pd = removendo_marca_tamanho_NA(df_pd)

    # Colunas de MetaDados (Hora da ação | Momento da ação)
    df_pd["Data_Linha"] = datetime.now()
    df_pd["LinhaOrigem"] = "SQLServer_DW_Dim_Produto"

    # Inserindo uma linha fake de df_produto

    # Criando dicionario com linha fake
    linha_df_pd_fake = {
        "ID_Produto": -1,
        "Nome_Produto": "Não Identificado",
        "Marca": "N/A",
        "Tamanho": "N/A",
        "Preco_Recomendado": -1,
        "Cor": "N/A",
        "Fornecedor": "N/A",
        "Categoria": "N/A", 
        "Subcategoria": "N/A",
        "Data_Linha": datetime(1900, 1, 1),
        "LinhaOrigem": "Aplicado Manualmente"
    }

    # Transformando dicionario em dataframe
    linha_df_pd_fake = pd.DataFrame([linha_df_pd_fake])

    # Adicionando a linha fake ao df_produto
    df_pd = pd.concat([df_pd, linha_df_pd_fake], ignore_index=True)
    
    # Retornando df_produto (Completo)
    return df_pd

### 👤 4.3. Dimensão: Cliente (Segmentação e Regras de Negócio)
Nesta etapa, focamos na qualificação dos dados da **Dimensão Cliente**. Implementamos uma lógica de Crédito para categorizar o perfil financeiro de cada cliente.

🛠️ Processos de Enriquecimento:
- **Segmentação de Perfil(`perfil_credito_col`)**: Criamos uma nova coluna `Perfil_Credito` baseada no limite de crédito do cliente.
    - **Tratamento de Crédito Ilimitado**: Conforme a documentação de origem, valores NaN (Nulos) indicam crédito ilimitado. Para fins de processamento e ordenação, esses valores são convertidos para `999999.999`.
    - **Classificação de Perfil**: A classificação é determinada em 4 tipos: Diamante(ilimitado), Ouro, Prata e Bronze, cada classificação sendo baseada no `Credito_Limite`.

- **Tradução de dados (`traducao_categoria_cliente`)**: Realizamos a tradução dos tipos de clientes para PT-BR utilizando o dicionário de referência em `.csv`.

In [56]:
# Nova Coluna Perfil_Credito
def perfil_credito_col(df_cl):
    # Clientes com Null Representam Creditos Ilimitados
    df_cl = df_cl.fillna(999999.999)

    # Condição para dados na nova coluna
    lista_condicao_perfil = [
        (df_cl["Credito_Limite"] == 999999.999),
        (df_cl["Credito_Limite"] >= 3000.000),
        (df_cl["Credito_Limite"] >= 1500.000)
    ]

    # Baseado nas condições esses serão as linhas a serem preenchidas
    escolhas_perfil = [
        "Diamante", # == 999999
        "Ouro",     # >= 3000
        "Prata"     # >= 1500
    ]

    # Nova coluna criada baseada no Credito_Limite
    # (Diamante, Ouro, Prata e Bronze)
    df_cl["Perfil_Credito"] = (np.select(lista_condicao_perfil, escolhas_perfil, default="Bronze"))
    
    # Retornando a df_cliente com uma nova coluna "Perfil_Credito"
    return df_cl

# Tradução para PT_BR da coluna "Categoria"
def traducao_categoria_cliente(df_cl):
    # Buscando CSV com traduções das categorias
    df_categoria_cliente = pd.read_csv("metadata/referencias-categoria-cliente.csv")

    # Transformando em um dicionario para usar no .map
    categoria_cliente = (dict(zip(
        df_categoria_cliente["categoria_original"],
        df_categoria_cliente["categoria_traduzida"])))

    # Aplicando a tradução as linhas da coluna "Categoria"
    df_cl["Categoria"] = df_cl["Categoria"].map(categoria_cliente)

    # Retornando o df_cliente com a Tradução
    return df_cl

### 📦 4.3.1. Dimensão: Cliente (Consolidação e Encapsulamento)
Nesta etapa, unificamos todas as funções de tratamento em um único fluxo de execução `carga_df_cliente`. Objetivo é garantir que todas as regras de negócio sejam aplicadas de forma sequencial e padronizadas.

🛠️ Ações Realizadas:
- **Segmentação Financeira**: Execução da lógica de perfil de crédito e a tradução das categorias de clientes.

- **Padrões de Segurança**: Inclusão de metadados (`Data_Linha` e `LinhaOrigem`) e Injeção de registros fake(`ID -1`).

- **Funções genéricas**: Aplicação das funções globais para o tratamentos de strings (`renomeando_col` e `tratamento_string`) e a tradução das colunas para o PT-BR.

In [57]:
# Tratamento df_cliente (Completa)
def carga_df_cliente(df_cl):
    # Tratando Strings de df_cliente
    df_cl = renomeando_col(df_cl, df_colunas_rename)
    df_cl = tratamento_string(df_cl)

    # Adição de nova Coluna Perfil_Credito
    df_cl = perfil_credito_col(df_cl)

    # Tradução das linhas "Categoria" para PT_BR
    df_cl = traducao_categoria_cliente(df_cl)

    # Colunas de MetaDados (Hora da ação | Momento da ação)
    df_cl["Data_Linha"] = datetime.now()
    df_cl["LinhaOrigem"] = "SQLServer_DW.Dim_Cliente"

    # Inserindo uma linha fake de df_cliente

    # Criando dicionario com linha fake
    linha_df_cl_fake = {
        "ID_Cliente": -1,
        "Nome_Cliente": "Não Identificado",
        "Credito_Limite": -1,
        "Numero_Telefone": "N/A",
        "Categoria": "N/A",
        "Perfil_Credito": "N/A",
        "Data_Linha": datetime(1900, 1, 1),
        "LinhaOrigem": "Aplicado Manualmente"
    }

    # Transformando dicionario em dataframe
    linha_df_cl_fake = pd.DataFrame([linha_df_cl_fake])

    # Adicionando a linha fake ao df_produto
    df_cl = pd.concat([df_cl, linha_df_cl_fake], ignore_index=True)
    
    # Retornando o df_cliente (Completo)
    return df_cl

### 👥 4.4. Dimensão: Vendedor (Regras de Negócio)
Nesta etapa, realizamos o ajuste fino na Dimensão Vendedor, focando na padronização logísticas dos metodos de Entrega vinculados a cada vendedor.

🛠️ Processos de Enriquecimento:
- **Logística(`Metodo_Entrega_vendedor`)**: Garantindo que nenhum registro fique sem seu `Metodo_Entrega`.

    - **Tratamento de Nulos**: Caso o método de entrega original seja **NaN (Nulo)**, a linha é preenchida automaticamente com o valor "**Transporte Padrão**".

In [58]:
# Método de entrega de cada Vendedor
def Metodo_Entrega_vendedor(df_vd):
    # Em Caso de dados NaN, teremos uma "Transporte Padrão" como entrega
    df_vd["Metodo_Entrega"] = (df_vd["Metodo_Entrega"]
                               .fillna("Transporte Padrão"))

    # Retornando a df_vendedor com Metodo_Entrega
    return df_vd

### 📦 4.4.1. Dimensão: Vendedor (Consolidação e Encapsulamento)
Nesta etapa, unificamos todas as funções de tratamento em um único fluxo de execução `carga_df_vendedor`. Objetivo é garantir que todas as regras de negócio sejam aplicadas de forma sequencial e padronizadas.

🛠️ Ações Realizadas:
- **Regras de Negócio**: Execução da função de preenchimento do método de entrega padrão para registros ausentes.

- **Padrões de Segurança**: Inclusão de metadados (`Data_Linha` e `LinhaOrigem`) e Injeção de registros fake(`ID -1`).

- **Funções genéricas**: Aplicação das funções globais para o tratamentos de strings (`renomeando_col` e `tratamento_string`) e a tradução das colunas para o PT-BR.

In [59]:
# Tratamento df_vendedor (Completa)
def carga_df_vendedor (df_vd):
    # Tratamentos de String df_vendedor
    df_vd = renomeando_col(df_vd, df_colunas_rename)
    df_vd = tratamento_string(df_vd)

    # Metodo_Entrega para cada Vendedor
    df_vd = Metodo_Entrega_vendedor(df_vd)

    # Colunas de MetaDados (Hora da ação | Momento da ação)
    df_vd["Data_Linha"] = datetime.now()
    df_vd["LinhaOrigem"] = "SQLServer_DW.Dim_Vendedor"

    # Inserindo uma linha fake de df_produto

    # Criando dicionario com linha fake
    linha_df_cl_fake = {
        "ID_Vendedor": -1,
        "Nome_Vendedor": "Não Identificado",
        "Email": "Não Identificado",
        "Metodo_Entrega": "N/A",
        "Data_Linha": datetime(1900, 1, 1),
        "LinhaOrigem": "Aplicado Manualmente"
    }

    # Transformando dicionario em dataframe
    linha_df_cl_fake = pd.DataFrame([linha_df_cl_fake])

    # Adicionando a linha fake ao df_produto
    df_vd = pd.concat([df_vd, linha_df_cl_fake], ignore_index=True)

    # Retornando df_vendedor (Completo)
    return df_vd

### 🌎 4.5. Dimensão: Geográfica (Geocodificação e Enriquecimento)
Nesta etapa, enriquecemos a **Dimensão Geográfica** buscando **latitude** e **longitude** para possibilitar análises em mapas. Utilizamos a biblioteca **Geopy** para converter nomes de estados em dados geográficos precisos.

🛠️ Processos de Enriquecimento:
- **Geocodificação(`latitude_longitude`)**: Implementamos uma integração com a **API Nominatim** para obter a **Latitude** e **Longitude**.

    - **Novas Colunas**: Como resultado desse processamento, o DataFrame é enriquecido com as colunas `Latitude` e `Longitude`, anteriormente inexistentes na base de dados.

    - **Limpeza com Regex**: Antes da consulta, aplicamos funções para remover parênteses **()** e colchetes **[]** dos nomes dos **estados**, garantindo que a API receba uma string limpa e aumente a taxa de sucesso na busca.

    - **Controle(Temporizador)**: Adicionamos um **delay de 1.2 segundos** e um **timeout de 10 segundos** entre cada chamada. Uma prática que evita qualquer bloqueio por exagerar no uso sem respeitar os limites.

    - **Tratamento de Erros**: O código foi envolvido em um `try/except`, caso a localização não seja encontrada, o sistema retorna valores padrão, evitando a interrupção do pipeline, e permitindo uma fácil identificação nas falhas no mapa.

In [60]:
# Configurando o geopy
# Nomeando o nosso agente de geopy (Nominatim)
geolocator = Nominatim(user_agent="Latitude-Longitude-Dim_Geografica")

# Função para o uso do Apply
def latitude_longitude (estado):
    # Tratativa de caso não encontre latitude e longitude!
    try:
        # Delay de 1.2 segundos antes de chamar API Geopy
        time.sleep(1.2)

        # Limpando o nome do "Estado", removendo parenteses e colchetes () []
        estado_limpo = re.sub(r"\(.*?\)|\[.*?\]", "", estado).strip()

        # Buscando a localização através do nome do "Estado"
        localizacao = geolocator.geocode(estado_limpo, timeout=10)

        # Através da localização buscada, pegamos Longitude e Latitude
        longitude = localizacao.longitude
        latitude = localizacao.latitude

        # Retornando uma List, usando o pd.Series para passar no df 2 colunas de uma vez
        return pd.Series([latitude, longitude])
    
    # Erro nos dados ou Latitude e Longitude não encontradas (inexistentes)
    except e:
        print(f"❌Erro no processamento {e}, ou latitude e longitude não encontrada")

        # Caso ele não encontre a latitude e longitude
        # Passaremos um inexistente
        return pd.Series([999.0, 999.0])

### 📦 4.5.1. Dimensão: Geográfica (Consolidação e Encapsulamento)
Nesta etapa, unificamos todas as funções de tratamento em um único fluxo de execução `carga_df_geografica`. O objetivo é garantir que todas as regras de negócio sejam aplicadas de forma sequencial e padronizada.

🛠️ Ações Realizadas:
- **Geolocalização**: Execução da função que consome a **API** para geração das coordenadas de Latitude e Longitude baseadas no estado.

- **Padrões de Segurança**: Inclusão de metadados (`Data_Linha` e `LinhaOrigem`) e Injeção de registros fake(`ID -1`).

- **Funções genéricas**: Aplicação das funções globais para o tratamentos de strings (`renomeando_col` e `tratamento_string`) e a tradução das colunas para o PT-BR.

In [61]:
# Tratamento df_geografica (Completa)
def carga_df_geografica(df_geo):
    # Tratando Strings de df_geografica
    df_geo = renomeando_col(df_geo, df_colunas_rename)
    df_geo = tratamento_string(df_geo)

    # Criando as 2 novas colunas, e passando latitude e longitude
    df_geo[["Latitude", "Longitude"]] = df_geo["Estado"].apply(latitude_longitude)

    # Colunas de MetaDados (Hora da ação | Momento da ação)
    df_geo["Data_Linha"] = datetime.now()
    df_geo["LinhaOrigem"] = "SQLServer_DW.Dim_Geografica"

    # Inserindo uma linha fake de df_geografica

    # Criando dicionario com linha fake
    linha_df_geo_fake = {
        "ID_Geografica": -1,
        "Estado": "Não Identificado",
        "Pais": "Não Identificado",
        "Latitude": 0.0,
        "Longitude": 0.0,
        "Data_Linha": datetime(1900, 1, 1),
        "LinhaOrigem": "Aplicado Manualmente"
    }

    # Transformando dicionario em dataframe
    linha_df_geo_fake = pd.DataFrame([linha_df_geo_fake])

    # Adicionando a linha fake ao df_produto
    df_geo = pd.concat([df_geo, linha_df_geo_fake], ignore_index=True)
    
    # Retornando df_geografico (Completo)
    return df_geo

### 📅 4.6. Dimensão: Data (Variaveis e Configuração)
Nesta etapa, preparamos os fundamentos para a criação da **Dimensão Data**. Diferente das dimensões anteriores, esta é uma dimensão criada do 0(zero), para cobrir todo o período de análise do projeto.

🛠️ Processos de Enriquecimento:
- **Começo e fim**: Variáveis `data_inicio` **(2011)** e `data_fim` **(2026)**, essa configuração permite que o DataFrame de datas seja facilmente ajustado caso o histórico de dados aumente.

- **Dicionario de Tradução**: Dicionarios `traducao_meses` e `traducao_dias` sendo um mapeamento dos nomes dos mesês e nome dos dias para o PT-BR.

- **DataFrame**: Inicialização de um DataFrame `df_data` que posteriormente servira como base para novos atributos temporais.

In [62]:
# Data (Inicio - Fim)
data_inicio = "2011"
data_fim = "2026"

# Criando o df_data (GLOBAL)
df_data = pd.DataFrame()

# Tradução dos dados da coluna Nome_Mes
traducao_meses = {
    "January": "Janeiro",
    "February": "Fevereiro",
    "March": "Março",
    "April": "Abril",
    "May": "Maio",
    "June": "Junho",
    "July": "Julho",
    "August": "Agosto",
    "September": "Setembro",
    "October": "Outubro",
    "November": "Novembro",
    "December": "Dezembro"
}

# Tradução dos dados da coluna Semana_Dia
traducao_dias = {
    "Monday": "Segunda-Feira",
    "Tuesday": "Terça-Feira",
    "Wednesday":"Quarta-Feira",
    "Thursday":"Quinta-Feira",
    "Friday":"Sexta-Feira",
    "Saturday":"Sabado",
    "Sunday":"Domingo"
}

### 📅 4.6.1. Dimensão: Data (Regras de Negócios)
Nesta etapa, preenchemos o DataFrame `df_data` gerando uma sequência contínua de datas. Utilizamos a `Data_Completa` para criação de uma Chave Primaria da dimensão Data, que posteriormente será relacionada com a coluna de datas da Fato.

🛠️ Transformações:

- **Geração de Datas**: Utilizamos `pd.date_range` para criar um intervalo ininterrupto entre as variáveis de `data_inicio` e `data_fim` definidas anteriormente.

- **Chave Primária (ID_Data)**: Criamos a **Chave Primaria** convertendo a data para o formato `YYYYMMDD` (ex: `2024-11-30` vira `20241130`) e transformando-a em dado do tipo **Inteiro**. Esse formato facilita o relacionamento com a Tabela Fato.

- **Enriquecimento de Atributos**: Extração de Ano, Mês, Dia, Trimestre, Semestre, Nome do Mês (PT-BR), Dia da Semana, Abreviação de Mês/Ano e o número da semana no ano.

- **Análise Calendária**: Criação da coluna `Fim_De_Semana` como um valor booleano, permitindo a segmentação rápida entre dias úteis e finais de semana para análises de performance de vendas.

- **Tradução de dados**: Tradução dos nomes dos meses e nomes dos dias da semana.

- **Padrões de Segurança**: Inclusão de metadados (`Data_Linha` e `LinhaOrigem`) e Injeção de registros fake(`ID -1`).

In [63]:
# Tratamento df_data (Completa)
def carga_df_data (df_dt):
    # Todas as datas de 2011 a 2026
    data_completa = pd.date_range(start=f"{data_inicio}-01-01", end=f"{data_fim}-12-31")

    # Transformando as datas em um DataFrame
    data_completa = pd.DataFrame(data_completa)

    # Mudando formato da data
    # %Y-%m-%d(2026-31-12) -> %Y%m%d(20263112)
    # data_completa[0] esse 0 representa o nome da coluna que foi criada sem definir um nome
    ID_Data = data_completa[0].dt.strftime('%Y%m%d')

    # Convertendo o ID_Data%Y%m%d(20263112) em um int
    df_dt["ID_Data"] = ID_Data.astype(int)

    # Adicionando coluna de Data_Completa ao df_data
    df_dt["Data_Completa"] = data_completa
 
    # Ano - Mes - Dia - Trimestre - Semestre
    df_dt["Ano"] = df_dt["Data_Completa"].dt.year
    df_dt["Mes"] = df_dt["Data_Completa"].dt.month
    df_dt["Dia"] = df_dt["Data_Completa"].dt.day
    df_dt["Trimestre"] = df_dt["Data_Completa"].dt.quarter

    # Template para buscar Semestre
    df_dt["Semestre"] = np.where(df_dt["Data_Completa"].dt.quarter <= 2, 1, 2)

    # Nomes: Mes - Dia - Mes/Ano
    df_dt["Nome_Mes"] = df_dt["Data_Completa"].dt.month_name().map(traducao_meses)
    df_dt["Semana_Dia"] = df_dt["Data_Completa"].dt.day_name().map(traducao_dias)
    df_dt["Mes_Ano"] = df_dt["Nome_Mes"].str[:3] + "/" + df_dt["Ano"].astype(str)

    # Numero da Semana
    df_dt["Dia_Semana"] = df_dt["Data_Completa"].dt.day_of_week + 1

    # Booleana para Final de Semana
    df_dt["Fim_De_Semana"] = np.where(df_dt["Dia_Semana"] <= 5, False, True)

    # Colunas de MetaDados (Hora da ação | Momento da ação)
    df_dt["Data_Linha"] = datetime.now()
    df_dt["LinhaOrigem"] = "SQLServer_DW.Dim_Data"

    # Inserindo uma linha fake de df_data

    # Criando dicionario com linha fake
    linha_df_pd_data = {
        "ID_Data": -1,
        "Data_Completa": datetime(1900, 1, 1),
        "Ano": 1900,
        "Mes": 1,
        "Dia": 1,
        "Trimestre": 1,
        "Semestre": 1,
        "Nome_Mes": "N/A", 
        "Semana_Dia": "N/A",
        "Mes_Ano": "N/A",
        "Dia_Semana": "1",
        "Fim_De_Semana": False,
        "Data_Linha": datetime(1900, 1, 1),
        "LinhaOrigem": "Aplicado Manualmente"
    }

    # Transformando dicionario em dataframe
    linha_df_pd_data = pd.DataFrame([linha_df_pd_data])

    # Adicionando a linha fake ao df_produto
    df_dt = pd.concat([df_dt, linha_df_pd_data], ignore_index=True)
    
    # Retornando df_data (Completa)
    return df_dt

### 📊 4.7. Fato: Vendas (Regras de Negócio)
Nesta etapa, processamos a tabela central do nosso modelo **Star Schema**. A Tabela Fato consolida os eventos de venda e realiza o cálculo de métricas essenciais para o negócio, além de garantir a integridade das **Chaves Estrangeiras**.

🛠️ Processos de Enriquecimento:
- **Alinhamento de Chave Temporal(`tratamento_ID_Data_Fato`)**: Realizamos a limpeza da coluna **ID_Data**, removendo hifens e convertendo o dado para o tipo **Inteiro**. Garantindo compatibilidade ao fazer o relacionamentos entre tabelas.

- **Cálculo de Métricas Financeiras(`colunas_fato_vendas`)**: Calculos realizados afins de buscar de buscar novas colunas, diferenciando diversos tipos de analises financeira.

    - **`Valor_Líquido_Total` e `Valor_Bruto_Total`**: Sendo o cálculo do montante total por item, <u>com</u> e <u>sem</u> impostos.

    - **`Custo`**: Representa o valor gasto para obter o **lucro**.

    - **`Imposto_Unitario`**: Montante do **imposto** sendo dividido pela **quantidade** de produtos.

    - **`Valor_Desconto`**: Cada produto possui um **valor de recomendação** de vendas, realizando uma subtração entre o **valor unitario**, conseguimos saber o **valor de desconto** que foi dado ao produto no momento da venda.

- **Lógica de Desconto**: Realizamos um **merge** temporário com a **Dimensão Produto** para obter o ``Preco_Recomendado``, subtraímos o **valor unitário** praticado do **preço recomendado**. Caso o valor de venda seja superior ao recomendado, o sistema atribui automaticamente 0 (zero) ao desconto, evitando margens negativas.

In [64]:
# Removendo hifen(-) do ID_Data, e convertendo a um Int
def tratamento_ID_Data_Fato(df_fv):
    # Removendo (-)
    df_fv["ID_Data"] = df_fv["ID_Data"].astype(str).str.replace("-", "")

    # Convertendo ID_Data a um Int
    df_fv["ID_Data"] = df_fv["ID_Data"].astype(int)

    # Retornando df_fatoVendas
    return df_fv


# Novas colunas para df_fatoVendas
def colunas_fato_vendas(df_fv, df_pd):
    # Imposto_Unitario (Imposto sobre cada quantidade de produto)
    df_fv["Imposto_Unitario"] = df_fv["Imposto_Pedido"] / df_fv["Quantidade"]

    # Valor total sem Imposto
    df_fv["Valor_Liquido_Total"] = df_fv["Quantidade"] * df_fv["Valor_Unitario"]

    # Valor total com Imposto
    df_fv["Valor_Bruto_Total"] = df_fv["Valor_Liquido_Total"] + df_fv["Imposto_Pedido"]

    # Custo
    df_fv["Custo"] = df_fv["Valor_Bruto_Total"] - df_fv["Imposto_Pedido"] - df_fv["Lucro"]

    # Desconto
    # Preco_Recomendado(df_produto)
    preco_recomendado = df_pd[["ID_Produto","Preco_Recomendado"]].copy()

    # Valor_Unitario(df_fatoVendas)
    valor_unitario = df_fv[["ID_Fatura", "ID_Produto", "Valor_Unitario"]].copy()

    # Aplicando Merge
    df_preco_recomendado_e_valor_unitario =(
        valor_unitario.merge(preco_recomendado, on="ID_Produto", how="left"))

    # Criando coluna Desconto
    df_fv["Valor_Desconto"] = (
        df_preco_recomendado_e_valor_unitario["Preco_Recomendado"] - df_preco_recomendado_e_valor_unitario["Valor_Unitario"])

    # Se o Valor_Unitario for maior que o Preco_Recomendado, vamos setar para 0(zero)
    df_fv["Valor_Desconto"] = np.where(df_fv["Valor_Desconto"] < 0, 0, df_fv["Valor_Desconto"])
    
    # Retornando df_fatoVendas
    return df_fv

### 📦 4.7.1. Fato: Vendas (Consolidação e Encapsulamento)
Nesta etapa, unificamos todas as funções de tratamento e regras de negócio no fluxo de execução `carga_df_fatoVendas`. O objetivo é consolidar a tabela central do **DW**, garantindo precisão e a integridade dos relacionamentos.

🛠️ Ações Realizadas:
- **Tratamento de Chaves Estrangeiras**: Implementamos uma camada de segurança utilizando `.fillna(-1)` em todas as colunas de **ID (Produto, Cliente, Vendedor, Data e Geográfica)**. 
Para evitar registros na Fato sem correspondência, eles são automaticamente vinculados aos nossos **"Registros Fake"** (`ID -1`), evitando a perda de dados e garantindo que o **Power BI/SQL** exiba esses valores como **"Não Identificado"**.

- **Padrões de Segurança**: Inclusão de metadados (`Data_Linha` e `LinhaOrigem`).

- **Funções genéricas**: Aplicação da função global para a tradução de colunas para PT-BR (`tratamento_string`).

In [65]:
# Tratamento df_fatovendas(Completa)
def carga_df_fatovendas(df_fv, df_pd):
    # Renomenado colunas de df_fatovendas
    df_fv = renomeando_col(df_fv, df_colunas_rename)

    # Tratamento do ID_Fato
    df_fv = tratamento_ID_Data_Fato(df_fv)

    # Em caso de FK Null, aplicaremos as FK fake com -1
    df_fv["ID_Produto"] = df_fv["ID_Produto"].fillna(-1)
    df_fv["ID_Cliente"] = df_fv["ID_Cliente"].fillna(-1)
    df_fv["ID_Vendedor"] = df_fv["ID_Vendedor"].fillna(-1)
    df_fv["ID_Data"] = df_fv["ID_Data"].fillna(-1)
    df_fv["ID_Geografica"] = df_fv["ID_Geografica"].fillna(-1)

    # Adicionando novas colunas
    df_fv = colunas_fato_vendas(df_fv, df_pd)

    # Colunas de MetaDados (Hora da ação | Momento da ação)
    df_fv["Data_Linha"] = datetime.now()
    df_fv["LinhaOrigem"] = "SQLServer_DW.Dim_Fato_Vendas"

    # Retornando df_fatovendas (Completa)
    return df_fv

## 🚀 5. Orquestração e Consolidação do Pipeline (Camada Silver)
Nesta etapa final do processamento, implementamos a função mestre `executar_tratamento_completo`. Ela atua como o orquestrador central que dispara todas as transformações de forma sequencial, garantindo que a **Camada Silver (Prata)** seja gerada com integridade.

🛠️ Inteligência do Pipeline:

- **Segurança**: Utilizamos o método `.copy()` para criar **variáveis locais** de cada **DataFrame**. Isso garante que os **dados brutos** permaneçam intactos caso ocorra algum erro durante o processamento.

    - **`Try`**: Se o **pipeline** rodar sem falhas, os **DataFrames** tratados são exportados para a pasta **data/silver** no formato `.csv`.

    - **`Except`**: Caso ocorra qualquer erro, o sistema captura a falha, detalha o erro e retorna os **DataFrames originais**, impedindo que dados corrompidos avancem para o **Data Warehouse**.

In [66]:
# Tratamento completo de todos Dataframe
def executar_tratamento_completo(df_pd, df_cl, df_vd, df_geo, df_dt, df_fv):
    # Variaveis locais(l = local) - Dataframe
    df_pd_l = df_pd.copy()     
    df_cl_l = df_cl.copy()     
    df_vd_l = df_vd.copy()     
    df_geo_l = df_geo.copy()     
    df_dt_l = df_dt.copy()     
    df_fv_l = df_fv.copy()    

    print("⚙️ Iniciando o processamento unificado das Dimensões e da Fato")
    print(".\n.\n.")

    # Tratamento de todos DataFrame...
    try:
        # Tratamento completo na df_produto
        df_pd_l = carga_df_produto(df_pd_l)
        print("✔️ df_produto tratada com Sucesso!")

        # Tratamento completo na df_cliente
        df_cl_l = carga_df_cliente(df_cl_l)
        print("✔️ df_cliente tratada com Sucesso!")

        # Tratamento completo na df_vendedor
        df_vd_l = carga_df_vendedor(df_vd_l)
        print("✔️ df_vendedor tratada com Sucesso!")

        # Tratamento completo na df_geografica
        df_geo_l = carga_df_geografica(df_geo_l)
        print("✔️ df_geografica tratada com Sucesso!")
        
        # Tratamento completo na df_data
        df_dt_l = carga_df_data(df_dt_l)
        print("✔️ df_data tratada com Sucesso!")

        # Tratamento completo na df_fatovendas
        df_fv_l = carga_df_fatovendas(df_fv_l, df_pd_l)
        print("✔️ df_fatovendas tratada com Sucesso!")

        # Exportando DataFrame tratados (Prata)
        print("\n💾 Exportando todos DataFrame.csv tratados para [data/silver] \n")
        df_pd_l.to_csv("data/silver/trst_produto.csv", index=False, encoding='utf-8-sig')
        df_cl_l.to_csv("data/silver/trst_cliente.csv", index=False, encoding='utf-8-sig')
        df_vd_l.to_csv("data/silver/trst_vendedor.csv", index=False, encoding='utf-8-sig')
        df_geo_l.to_csv("data/silver/trst_geografica.csv", index=False, encoding='utf-8-sig')
        df_dt_l.to_csv("data/silver/trst_data.csv", index=False, encoding='utf-8-sig')
        df_fv_l.to_csv("data/silver/trst_fatovendas.csv", index=False, encoding='utf-8-sig')

        # Mensagem de Sucesso
        print(".\n.\n.")
        print("\n Status: Sucesso")
        print("✔️ Todos DataFrame foram analisados e tratados com sucesso!")
        print("📚 ​Retornando todos os DataFrames alterados, SIM ESTÃO PRONTOS para Data Warehouse!")

        df_pd_l = df_pd_l.drop_duplicates(subset=["ID_Produto"])
        df_cl_l = df_cl_l.drop_duplicates(subset=["ID_Cliente"])
        df_vd_l = df_vd_l.drop_duplicates(subset=["ID_Vendedor"])
        df_geo_l = df_geo_l.drop_duplicates(subset=["ID_Geografica"])
        df_dt_l = df_dt_l.drop_duplicates(subset=["ID_Data"])


        # Retornando todos os Daframe em uma tupla (Ordem é importante)
        # Retornando Dataframes que funcionaram!
        return (df_pd_l, df_cl_l, df_vd_l, df_geo_l, df_dt_l, df_fv_l)

    # Detalhamento do erro
    except Exception as e: 
        print(".\n.\n.")
        print(f"❌ERRO: pipeline teve um problema! ⚠️ erro: {e}")
        print("📚 Retornando os Dataframes SEM alterações, NÃO ESTÃO PRONTOS pro Data Warehouse")

        # Retornando os dataframe originais (dados brutos)
        # Retornando Dataframes que não funcionaram
        return (df_pd, df_cl, df_vd, df_geo, df_dt, df_fv)

## 🔌 6. Carga e Persistência no Data Warehouse (Camada Ouro)
Nesta etapa final, os dados processados na **Camada Silver** são carregados para o banco de destino **(DW_VENDAS)**. Implementamos uma função reutilizável que gerencia a conexão e garante a integridade da carga através de comandos avançados de **SQL**.

🛠️ Conexão e Carga:
- **Conexão Segura**: Utilizamos o **SQLAlchemy** para estabelecer a ponte com o **SQL Server**, validando a disponibilidade do banco.

- **Tabela Temporária**: Em vez de inserir diretamente na tabela final, o **DataFrame** é carregado em uma tabela temporária (stg_nome). Isso permite uma validação aos dados, antes de qualquer alteração no **DW**.

- **MERGE (Upsert)**: Executamos uma **Query** de **MERGE** que realiza a adição de novas informações (`INSERT`) e a atualização de registros existentes (`UPDATE`). Essa técnica evita a duplicidade de dados e garante que o **Data Warehouse** persista sempre a versão mais recente das informações.

- **Gestão de Falhas (Rollback e `Raise`)**: O sistema utiliza um bloco `try/except` com interrupção de imediato em caso de algum erro.

- **`Finally`**: Garante a limpeza do ambiente, removendo a tabela temporaria, mesmo que tenha sucesso ou falha.

In [67]:
# Conectando ao Banco de Dados (Data Warehouse)
try:
    print(f"🔌 Conectando ao banco {databaseDestino}...")
    
    # Conectando ao Banco DW_VENDAS
    engine_destino = create_engine(conexao_destino)

    # Testando conexão
    with engine_destino.connect():
        print(f"✔️ Conexão ao {databaseDestino} com Sucesso!")
except Exception as e:
    print(f"❌ ERRO DE CONEXÃO{e}")


# Conexão | Inserção | Evitar duplicidade
def conexao_DW_merge(df, stg_nome, query_Merge, conexao_ativa):
    try:

        # Criando uma tabela temporaria no SQLServer
        df.to_sql(
            name=stg_nome,
            schema="DW",
            con=conexao_ativa,
            if_exists='replace',
            index=False
        )
        
        # Executando o Query no SQL
        conexao_ativa.execute(query_Merge)
        print("🤝 MERGE executado com Sucesso!")
        print("✔️ Inserção ou Atualização de dados, realizada com Sucesso")

    except Exception as e:
        # Em caso de erro, toda alteração sera desfeita
        print(f"❌ ERRO: {e}")

        print("⚠️​ Disparando RAISE!")
        raise e
    finally:
        # Apagando a tabela temporaria!
        conexao_ativa.execute(text(f"DROP TABLE IF EXISTS DW.{stg_nome}"))

🔌 Conectando ao banco DW_VENDAS...
✔️ Conexão ao DW_VENDAS com Sucesso!


### 🛠️​ 6.1. Arquitetura de Carga: Funções de Persistência
Devido à natureza única de cada entidade (Diferentes colunas, tipos de dados e chaves), desenvolvemos funções de carga individuais para cada tabela do Data Warehouse. Isso garante que o processo de **MERGE** seja executado com precisão para cada dimensão e para a tabela fato.

🛠️ Destaques da Implementação:
- **Execução do Merge**: Todas as funções (`Carregando_Dim_Produto_DW`, `Carregando_Dim_Cliente_DW`, etc.) utilizam a função `conexao_DW_merge` para a execução de cada query, correspondente a cada dimensão ou Fato.

- **Isolamento das tabelas (Stage)**: Cada função define um nome único para cada operação na tabela temporária(`stage_nome`), evitando conflitos de memória.

- **Lógica do Merge (`query_Merge_nome`)**: Basicamente atualizamos o registro caso sua chave primaria se repita e possua alguma coluna com algum registro diferente. E em caso de novos registros, serão adicionados ao Data Warehouse. Isso evita duplicidade ou ações desnecessárias, otimizando seu processo.

    - **Cláusula `WHEN MATCHED`**: Realiza um `UPDATE` caso um registro já existente no **Data Warehouse** tenha sofrido alguma alteração na origem.

    - **Cláusula `WHEN NOT MATCHED`**: Realiza um `INSERT` sempre que um novo registro for identificado.

In [68]:
# Carregando DW.Dim_Produto (DataWarehouse)
def Carregando_Dim_Produto_DW (df_pd, conexao_ativa):
        
    # Nome da tabela temporaria
    stage_nome = "stg_produto"

    # Query - Se já existir a gente altera em caso de alguma mudança - Caso não exista, adicionamos
    query_Merge_produto = text(
    f"""
        MERGE INTO DW.Dim_Produto as d
        USING DW.{stage_nome} as o
        ON (d.ID_Produto = o.ID_Produto)
        WHEN MATCHED 
        AND 
        (   o.Nome_Produto <> d.Nome_Produto OR d.Marca <> o.Marca OR 
            d.Tamanho <> o.Tamanho OR d.Preco_Recomendado <> o.Preco_Recomendado OR 
            o.Cor <> d.Cor OR o.Fornecedor <> d.Fornecedor OR 
            o.Categoria <> d.Categoria OR o.Subcategoria <> d.Subcategoria
        )
        THEN
        UPDATE SET 
        d.Nome_Produto = o.Nome_Produto, d.Marca = o.Marca, d.Tamanho = o.Tamanho,
        d.Preco_Recomendado = o.Preco_Recomendado, d.Cor = o.Cor, d.Fornecedor = o.Fornecedor, 
        d.Categoria = o.Categoria, d.Subcategoria = o.Subcategoria
        
        WHEN NOT MATCHED THEN
        INSERT (ID_Produto, Nome_Produto, Marca, Tamanho, Preco_Recomendado, Cor, Fornecedor, Categoria, Subcategoria, Data_Linha, LinhaOrigem)
        VALUES (o.ID_Produto, o.Nome_Produto, o.Marca, o.Tamanho, o.Preco_Recomendado, o.Cor, o.Fornecedor, o.Categoria, o.Subcategoria, o.Data_Linha, o.LinhaOrigem);
    """
    )

    # Chamando a função responsavel pela conexão, criação da tabela temporaria, e inserção de dados
    conexao_DW_merge(df_pd, stage_nome, query_Merge_produto, conexao_ativa)


# Carregando DW.Dim_Cliente (DataWarehouse)
def Carregando_Dim_Cliente_DW (df_cl, conexao_ativa):
        
    # Nome da tabela temporaria
    stage_nome = "stg_cliente"

    # Query - Se já existir a gente altera em caso de alguma mudança - Caso não exista, adicionamos
    query_Merge_cliente = text(
    f"""
        MERGE INTO DW.Dim_Cliente as d
        USING DW.{stage_nome} as o
        ON (d.ID_Cliente = o.ID_Cliente)
        WHEN MATCHED 
        AND (
            o.Nome_Cliente <> d.Nome_Cliente OR d.Credito_Limite <> o.Credito_Limite OR 
            d.Numero_Telefone <> o.Numero_Telefone OR d.Categoria <> o.Categoria OR
            o.Perfil_Credito <> d.Perfil_Credito
            )
        THEN
        UPDATE SET 
            d.Nome_Cliente = o.Nome_Cliente, d.Credito_Limite = o.Credito_Limite, 
            d.Numero_Telefone = o.Numero_Telefone, d.Categoria = o.Categoria,
            d.Perfil_Credito = o.Perfil_Credito
        WHEN NOT MATCHED THEN
        INSERT (ID_Cliente, Nome_Cliente, Credito_Limite, Numero_Telefone, Categoria, Perfil_Credito, Data_Linha, LinhaOrigem)
        VALUES (o.ID_Cliente, o.Nome_Cliente, o.Credito_Limite, o.Numero_Telefone, o.Categoria, o.Perfil_Credito, o.Data_Linha, o.LinhaOrigem);
    """
    )

    # Chamando a função responsavel pela conexão, criação da tabela temporaria, e inserção de dados
    conexao_DW_merge(df_cl, stage_nome, query_Merge_cliente, conexao_ativa)


# Carregando DW.Dim_Vendedor (DataWarehouse)
def Carregando_Dim_Vendedor_DW (df_vd, conexao_ativa):
        
    # Nome da tabela temporaria
    stage_nome = "stg_vendedor"

    # Query - Se já existir a gente altera em caso de alguma mudança - Caso não exista, adicionamos
    query_Merge_vendedor = text(
    f"""
        MERGE INTO DW.Dim_Vendedor as d
        USING DW.{stage_nome} as o
        ON (d.ID_Vendedor = o.ID_Vendedor)
        WHEN MATCHED 
        AND (o.Nome_Vendedor <> d.Nome_Vendedor OR d.Email <> o.Email OR d.Metodo_Entrega <> o.Metodo_Entrega)
        THEN
        UPDATE SET 
        d.Nome_Vendedor = o.Nome_Vendedor, d.Email = o.Email, d.Metodo_Entrega = o.Metodo_Entrega
        WHEN NOT MATCHED THEN
        INSERT (ID_Vendedor, Nome_Vendedor, Email, Metodo_Entrega, Data_Linha, LinhaOrigem)
        VALUES (o.ID_Vendedor, o.Nome_Vendedor, o.Email, o.Metodo_Entrega, o.Data_Linha, o.LinhaOrigem);
    """
    )

    # Chamando a função responsavel pela conexão, criação da tabela temporaria, e inserção de dados
    conexao_DW_merge(df_vd, stage_nome, query_Merge_vendedor, conexao_ativa)


# Carregando DW.Dim_Geografica (DataWarehouse)
def Carregando_Dim_Geografica_DW (df_geo, conexao_ativa):
    # Nome da tabela temporaria
    stage_nome = "stg_geografica"

    # Query - Se já existir a gente altera em caso de alguma mudança - Caso não exista, adicionamos
    query_Merge_geografica = text(
    f"""
        MERGE INTO DW.Dim_Geografica as d
        USING DW.{stage_nome} as o
        ON (d.ID_Geografica = o.ID_Geografica)
        WHEN MATCHED 
        AND (
            o.Estado <> d.Estado OR d.Pais <> o.Pais OR 
            d.Latitude <> o.Latitude OR d.Longitude <> o.Longitude
            )
        THEN
        UPDATE SET 
            d.Estado = o.Estado, d.Pais = o.Pais, 
            d.Latitude = o.Latitude, d.Longitude = o.Longitude
        WHEN NOT MATCHED THEN
        INSERT (ID_Geografica, Estado, Pais, Latitude, Longitude, Data_Linha, LinhaOrigem)
        VALUES (o.ID_Geografica, o.Estado, o.Pais, o.Latitude, o.Longitude, o.Data_Linha, o.LinhaOrigem);
    """
    )

    # Chamando a função responsavel pela conexão, criação da tabela temporaria, e inserção de dados
    conexao_DW_merge(df_geo, stage_nome, query_Merge_geografica, conexao_ativa)


# Carregando DW.Dim_Data (DataWarehouse)
def Carregando_Dim_Data_DW (df_dt, conexao_ativa):
        
    # Nome da tabela temporaria
    stage_nome = "stg_data"

    # Query - Se já existir a gente altera em caso de alguma mudança - Caso não exista, adicionamos
    query_Merge_data = text(
    f"""
        MERGE INTO DW.Dim_Data as d
        USING DW.{stage_nome} as o
        ON (d.ID_Data = o.ID_Data)
        WHEN MATCHED 
        AND 
        (   o.Data_Completa <> d.Data_Completa OR d.Ano <> o.Ano OR 
            d.Mes <> o.Mes OR o.Dia <> d.Dia OR
            o.Semestre <> d.Semestre OR o.Trimestre <> d.Trimestre OR
            o.Nome_Mes <> d.Nome_Mes OR o.Semana_Dia <> d.Semana_Dia OR
            o.Mes_Ano <> d.Mes_Ano OR o.Dia_Semana <> d.Dia_Semana OR
            o.Fim_De_Semana <> d.Fim_De_Semana
        )
        THEN
        UPDATE SET 
            d.Data_Completa = o.Data_Completa, d.Ano = o.Ano, d.Mes = o.Mes,
            d.Dia = o.Dia, d.Semestre = o.Semestre, d.Trimestre = o.Trimestre,
            d.Nome_Mes = o.Nome_Mes, d.Semana_Dia = o.Semana_Dia, d.Mes_Ano = o.Mes_Ano,
            d.Dia_Semana = o.Dia_Semana, d.Fim_De_Semana = o.Fim_De_Semana
        
        WHEN NOT MATCHED THEN
        INSERT (ID_Data, Data_Completa, Ano, Mes, Dia, Semestre, Trimestre, Nome_Mes, Semana_Dia, Mes_Ano, Dia_Semana, Fim_De_Semana, Data_Linha, LinhaOrigem)
        VALUES (o.ID_Data, o.Data_Completa, o.Ano, o.Mes, o.Dia, o.Semestre, o.Trimestre, o.Nome_Mes, o.Semana_Dia, o.Mes_Ano, o.Dia_Semana, o.Fim_De_Semana, o.Data_Linha, o.LinhaOrigem);
    """
    )

    # Chamando a função responsavel pela conexão, criação da tabela temporaria, e inserção de dados
    conexao_DW_merge(df_dt, stage_nome, query_Merge_data, conexao_ativa)


# Carregando DW.Fato_Vendas (DataWarehouse)
def Carregando_Fato_Vendas_DW (dt_fv, conexao_ativa):
        
    # Nome da tabela temporaria
    stage_nome = "stg_fatoVendas"

    # Query - Se já existir a gente altera em caso de alguma mudança - Caso não exista, adicionamos
    query_Merge_Fato = text(
    f"""
        MERGE INTO DW.Fato_Vendas as d
        USING DW.{stage_nome} as o
        ON (d.ID_Fatura = o.ID_Fatura)
        WHEN MATCHED 
        AND 
        (   o.ID_Produto <> d.ID_Produto OR d.ID_Cliente <> o.ID_Cliente OR 
            d.ID_Vendedor <> o.ID_Vendedor OR o.ID_Geografica <> d.ID_Geografica OR
            o.ID_Data <> d.ID_Data OR o.Quantidade <> d.Quantidade OR
            o.Valor_Unitario <> d.Valor_Unitario OR o.Lucro <> d.Lucro OR
            o.Valor_Liquido_Total <> d.Valor_Liquido_Total OR o.Imposto_Pedido <> d.Imposto_Pedido OR
            o.Custo <> d.Custo OR o.Valor_Bruto_Total = d.Valor_Bruto_Total OR 
            o.Imposto_unitario <> d.Imposto_unitario OR o.Valor_Desconto <> d.Valor_Desconto
        )
        THEN
        UPDATE SET 
            d.ID_Produto = o.ID_Produto, 
            d.ID_Cliente = o.ID_Cliente, 
            d.ID_Vendedor = o.ID_Vendedor,
            d.ID_Geografica = o.ID_Geografica, 
            d.ID_Data = o.ID_Data, 
            d.Quantidade = o.Quantidade,
            d.Valor_Unitario = o.Valor_Unitario, 
            d.Lucro = o.Lucro, 
            d.Valor_Liquido_Total = o.Valor_Liquido_Total,
            d.Imposto_Pedido = o.Imposto_Pedido, 
            d.Custo = o.Custo, 
            d.Valor_Bruto_Total = o.Valor_Bruto_Total,
            d.Imposto_unitario = o.Imposto_unitario,
            d.Valor_Desconto = o.Valor_Desconto
        
        WHEN NOT MATCHED THEN
        INSERT (ID_Fatura, ID_Produto, ID_Cliente, ID_Vendedor, ID_Geografica, ID_Data, Quantidade, Valor_Unitario, Lucro, Valor_Liquido_Total, Imposto_Pedido, Custo, Valor_Bruto_Total, Imposto_unitario, Valor_Desconto, Data_Linha, LinhaOrigem)
        VALUES (o.ID_Fatura, o.ID_Produto, o.ID_Cliente, o.ID_Vendedor, o.ID_Geografica, o.ID_Data, o.Quantidade, o.Valor_Unitario, o.Lucro, o.Valor_Liquido_Total, o.Imposto_Pedido, o.Custo, o.Valor_Bruto_Total, o.Imposto_unitario, o.Valor_Desconto, o.Data_Linha, o.LinhaOrigem);
    """
    )

    # Chamando a função responsavel pela conexão, criação da tabela temporaria, e inserção de dados
    conexao_DW_merge(dt_fv, stage_nome, query_Merge_Fato, conexao_ativa)

### 📝 6.2. Monitoramento (Logs de Carga)
Para garantir o controle total sobre o que acontece no **Data Warehouse**, implementamos uma função de monitoramento (`Carga_dim_fato_e_criacao_df_Log`). Esta camada é responsável por registrar cada transação, registrando métricas de performance e volume de dados.

🛠️ Ações do Monitoramento:

- **Captação em Tempo Real**: A função captura o exato momento do Início e Fim de cada carga, permitindo identificar gargalos de performance. 

    - **Tempo de execução**: O tempo de execução ele é definido dentro do SQL Server, o **DW.Log_DW_VENDAS** possui uma coluna (`Duracao_Data`) que é atribuido à diferença entre `Inicio_Data` e `Fim_Data`, utiliza o calculo: `Duracao_Data AS DATEDIFF(SECOND, Inicio_Data, Fim_Data)`.

- **Controle de Volume**: O sistema realiza um `COUNT` na tabela de destino <u>antes</u> e <u>depois</u> da operação. Com isso conseguimos mensurar exatamente quantas novas linhas foram inseridas no **DW**.

- **Mecanismo de Erro**: A função utiliza um bloco `try/except` que só registra o status como "**SUCESSO**" se a carga for **concluída 100%**. Caso ocorra uma única falha, o `raise` interrompe o processo imediatamente, garantindo que o log seja persistido com o detalhamento da falha.


In [69]:
# Criando log e processando carga nas Dimensões e Fato
def Carga_dim_fato_e_criacao_df_Log(nometabela, dataframe, funcao_carga_df, conexao):
    
    # Criando lista de colunas
    colunas = ["Tabela", "Inicio_Data", "Fim_Data", "Status", "Linhas", "Erro"]

    # Criando um dataframe com as colunas da list, porém com um index=[0] para destravar seu uso
    df_log = pd.DataFrame(columns=colunas, index=[0])

    # Adicionando nome da Tabela a Coluna "Tabela"
    df_log["Tabela"] = nometabela

    # Adicionando o tempo inicio a Coluna "Inicio_Data"
    df_log["Inicio_Data"] = datetime.now()

    # Em caso de Sucesso
    try:
        # Buscando a quantidade de linhas existentes na tabela X do DataWarehouse
        linha_antes = int(pd.read_sql(f"SELECT COUNT (*) AS Total FROM {nometabela}", conexao).iloc[0,0])

        # Processando carga no DataWarehouse, função sendo passada por parametro
        funcao_carga_df(dataframe, conexao)

        # Adicionando o tempo final a Coluna "Fim_Data"
        df_log["Fim_Data"] = datetime.now()

        # Buscando a quantidade de linhas existentes na tabela X do DataWarehouse **DEPOIS DA CARGA**
        linha_depois = int(pd.read_sql(f"SELECT COUNT (*) AS Total FROM {nometabela}", conexao).iloc[0,0])

        # Adicionando "SUCESSO" a Coluna "Status"
        df_log["Status"] = "SUCESSO"

        # Adicionando a quantidade de linhas alteradas se baseando no antes e depois a Coluna "Linhas"
        df_log["Linhas"] = linha_depois - linha_antes

        # Adicionando um valor qualquer a Coluna de "Erro" em caso de Sucesso
        df_log["Erro"] = "0"

        # Processando as informações e retornando a log
        return df_log
    
    except Exception as e:
        # Se der erro na carga, o Python pula direto para cá!
        print(f"❌ Falha detectada na tabela: {nometabela}")
        
        # Imprimindo erro
        print("⚠️​ Disparando RAISE!")
        raise e

## 🚀 7. Execução Final e Segurança do Pipeline
Esta é a etapa mestre do projeto, onde consolidamos todas as funções desenvolvidas anteriormente em um fluxo único. O objetivo aqui é garantir que a transição da **Camada Silver** para a **Camada Ouro** (**DW**) ocorra tudo bem.

🛠️ Inteligência do Pipeline:

- **Transacional**: Utilizamos o controle de transações do **SQL Server** (`conn.begin()`). Tudo ou nada, o `commit` final só é disparado após a carga de todas as dimensões, da tabela fato e seus logs de sucesso. Se qualquer tabela falha, nada é persistido.

- **Segurança**: Um `rollback` de segurança é disparado em caso de erro em qualquer etapa, o sistema executa automaticamente um `rollback`, desfazendo qualquer alteração e protegendo o **Data Warehouse** de dados inconsistentes.

- **Diagnóstico**: Se o **pipeline** falhar, o sistema captura a exceção e gera um **Log de Falha** específico, registrando o erro técnico no banco de dados.
    - **Envio do Log (Falha)**: Em caso de **falha**, o **log de erro** é enviado usando o `engine_destino` diretamente, em vez da conexão que sofreu o **rollback**. Isso garante que o registro da falha seja gravado no banco de dados sem a necessidade do `commit` final, impedindo que o erro seja descartado junto com as operações de carga que falharam.

- **Encerramento**: O uso do bloco `finally` garante que a conexão com o servidor seja encerrada (`close`) independentemente do resultado. 

In [70]:
# Todos DataFrames como Parametros
def Carregamento_dw_completo(df_pd, df_cl, df_vd, df_geo, df_dt, df_fv):
    print("\n📚​ Iniciando o Pipeline de dados...\n")

    try: 
        print("🔌 Conectando ao Data Warehouse")
        # Abrindo a conexão manualmente
        conn = engine_destino.connect()

        # Bloco de monitoramento (Transacional)
        trans = conn.begin()

        # DW.Dim_Produto
        pd_log = Carga_dim_fato_e_criacao_df_Log("DW.Dim_Produto", df_pd, Carregando_Dim_Produto_DW, conn)
        print("✔️​ DW.Dim_Produto\n\n")

        # DW.Dim_Cliente
        cl_log = Carga_dim_fato_e_criacao_df_Log("DW.Dim_Cliente", df_cl, Carregando_Dim_Cliente_DW, conn)
        print("✔️​ DW.Dim_Cliente\n\n")

        # DW.Dim_Vendedor
        vd_log = Carga_dim_fato_e_criacao_df_Log("DW.Dim_Vendedor", df_vd, Carregando_Dim_Vendedor_DW, conn)
        print("✔️​ DW.Dim_Vendedor\n\n")

        # DW.Dim_Geografica
        geo_log = Carga_dim_fato_e_criacao_df_Log("DW.Dim_Geografica", df_geo, Carregando_Dim_Geografica_DW, conn)
        print("✔️​ DW.Dim_Geografica\n\n")

        # DW.Dim_Data
        dt_log = Carga_dim_fato_e_criacao_df_Log("DW.Dim_Data", df_dt, Carregando_Dim_Data_DW, conn)
        print("✔️​ DW.Dim_Data\n\n")

        # DW.Fato_Vendas
        fv_log = Carga_dim_fato_e_criacao_df_Log("DW.Fato_Vendas", df_fv, Carregando_Fato_Vendas_DW, conn)
        print("✔️​ DW.Fato_Vendas\n\n")

        # Log de Sucesso

        # Log de Produto
        pd_log.to_sql(name="Log_DW_VENDAS", schema="DW", con=conn, if_exists='append', index=False)
    
        # Log de Cliente
        cl_log.to_sql(name="Log_DW_VENDAS", schema="DW", con=conn, if_exists='append', index=False)

        # Log de Vendedor
        vd_log.to_sql(name="Log_DW_VENDAS", schema="DW", con=conn, if_exists='append', index=False)

        # Log de Geografica
        geo_log.to_sql(name="Log_DW_VENDAS", schema="DW", con=conn, if_exists='append', index=False)

        # Log de Data
        dt_log.to_sql(name="Log_DW_VENDAS", schema="DW", con=conn, if_exists='append', index=False)

        # Log de Fato_Vendas
        fv_log.to_sql(name="Log_DW_VENDAS", schema="DW", con=conn, if_exists='append', index=False)

        # Subindo todos os Log para o banco de dados
        trans.commit()

    except Exception as e:
        print(f"❌ ERRO no Pipeline: {e}")

        # Rollback em caso de erro
        trans.rollback()

        # Dicionario de Log Erro
        dados_erro = {
            "Tabela": ["ERRO_NO_PIPELINE"],
            "Inicio_Data": [datetime.now()],
            "Fim_Data": [datetime.now()],
            "Status": ["FALHA"],
            "Linhas": [0],
            "Erro": [str(e)] 
        }

        # Transformando dicionario de Log erro em um Dataframe
        log_erro = pd.DataFrame(dados_erro)

        # Subindo o Log erro para a Log_DW_VENDAS do Banco de dados
        log_erro.to_sql(name="Log_DW_VENDAS", schema="DW", con=engine_destino, if_exists='append', index=False)

    finally:
         # Fechando a conexão automaticamente
        conn.close()
        print(f"🔌🔒 Conexão com {databaseDestino} encerrada!")

## 🏁 8. Execução Pricipal do Pipeline (Fim a Fim)
Nesta etapa final, executamos a integração total do projeto. O fluxo segue a ordem lógica de dependência dos dados: primeiro o tratamento das informações (**Camada Silver**) e, em seguida, a carga segura no **Data Warehouse** (**Camada Ouro**).

⚙️ Ações:

1. **DataFrames Tratados**: Criação de 6 **DataFrames**, onde serão armazenados os dados já tratados. Isso garante um **DataFrame** limpo para os tratamentos, evitando que re-execuções acidentais acumulem lixo ou dupliquem dados na memória.

2. **Tratamento de Dados**: Chamamos a função `executar_tratamento_completo`, que processa as tabelas originais e nos retorna os DataFrames prontos.

3. **Carga no Data Warehouse**: Com os dados tratados em mãos, disparamos a função `Carregamento_dw_completo`. Ela gerencia a transação, executa os Merges e registra os logs de **sucesso** ou **falha**.

4. **Encerramento**: Realizamos o encerramento do **engine** do **DW**.

In [71]:
# DataFrames final
df_produto_final = pd.DataFrame()
df_cliente_final = pd.DataFrame()
df_vendedor_final = pd.DataFrame()
df_geografica_final = pd.DataFrame()
df_data_final = pd.DataFrame()
df_fatovendas_final = pd.DataFrame()

In [72]:
# Não será executado ao importar
if __name__ == "__main__":
    # Executando tratamento dos DataFrames
    df_produto_final, df_cliente_final, df_vendedor_final, df_geografica_final, df_data_final, df_fatovendas_final = (
        executar_tratamento_completo(df_produto, df_cliente, df_vendedor, df_geografica, df_data, df_fatovendas)
    )

⚙️ Iniciando o processamento unificado das Dimensões e da Fato
.
.
.
✔️ df_produto tratada com Sucesso!
✔️ df_cliente tratada com Sucesso!
✔️ df_vendedor tratada com Sucesso!
✔️ df_geografica tratada com Sucesso!
✔️ df_data tratada com Sucesso!
✔️ df_fatovendas tratada com Sucesso!

💾 Exportando todos DataFrame.csv tratados para [data/silver] 

.
.
.

 Status: Sucesso
✔️ Todos DataFrame foram analisados e tratados com sucesso!
📚 ​Retornando todos os DataFrames alterados, SIM ESTÃO PRONTOS para Data Warehouse!


In [73]:
# Não será executado ao importar
if __name__ == "__main__":
    # Inserindo dataframes no Data Warehouse
    Carregamento_dw_completo(df_produto_final, df_cliente_final, df_vendedor_final, df_geografica_final, df_data_final, df_fatovendas_final)


📚​ Iniciando o Pipeline de dados...

🔌 Conectando ao Data Warehouse
🤝 MERGE executado com Sucesso!
✔️ Inserção ou Atualização de dados, realizada com Sucesso
✔️​ DW.Dim_Produto


🤝 MERGE executado com Sucesso!
✔️ Inserção ou Atualização de dados, realizada com Sucesso
✔️​ DW.Dim_Cliente


🤝 MERGE executado com Sucesso!
✔️ Inserção ou Atualização de dados, realizada com Sucesso
✔️​ DW.Dim_Vendedor


🤝 MERGE executado com Sucesso!
✔️ Inserção ou Atualização de dados, realizada com Sucesso
✔️​ DW.Dim_Geografica


🤝 MERGE executado com Sucesso!
✔️ Inserção ou Atualização de dados, realizada com Sucesso
✔️​ DW.Dim_Data


🤝 MERGE executado com Sucesso!
✔️ Inserção ou Atualização de dados, realizada com Sucesso
✔️​ DW.Fato_Vendas


🔌🔒 Conexão com DW_VENDAS encerrada!


In [75]:
# 4.
# Finalizando todas conexões
def fecharConexoes():
    engine_origem.dispose()
    engine_destino.dispose()
    print(f"🔌🔒 ​Conexão geral encerrada!")

# Chamando a função para finalizar
# Não será executado ao importar
if __name__ == "__main__":
    fecharConexoes()

🔌🔒 ​Conexão geral encerrada!
